# Multi Query Retriever — 쿼리 다각화

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 단일 쿼리 검색의 한계와 **MultiQueryRetriever**가 이를 보완하는 원리 이해하기
- `MultiQueryRetriever.from_llm()`으로 LLM 기반 쿼리 다각화 구현하기
- 로깅으로 LLM이 생성한 쿼리 목록을 직접 확인하기

## 사전 지식

- VectorStoreRetriever 기본 사용법
- LLM이 텍스트를 이해하고 변환하는 원리

---

## 핵심 아이디어

하나의 쿼리를 LLM이 여러 관점으로 재구성해요:

| | 내용 |
|---|---|
| **원본 쿼리** | "딥러닝이 뭐야?" |
| **변환 쿼리 1** | "딥러닝의 정의는 무엇인가?" |
| **변환 쿼리 2** | "딥러닝 기술에 대해 설명해주세요" |
| **변환 쿼리 3** | "딥러닝은 어떤 학습 방법인가?" |

각 쿼리로 검색 후 결과를 통합(중복 제거)합니다.

> **주의**: MultiQueryRetriever는 LLM을 추가로 호출하기 때문에 응답 속도가 느려지고 비용이 증가해요. 검색 품질이 중요한 경우에 선택적으로 사용하세요.

> **실무 팁**: 로깅 레벨을 INFO로 설정하면 LLM이 생성한 쿼리들을 콘솔에서 확인할 수 있어요.

> 🔑 **핵심 개념**: 벡터 검색은 쿼리 표현 방식에 민감합니다. 같은 의미라도 다르게 표현하면 다른 문서가 검색되어요. MultiQueryRetriever는 LLM을 활용해 자동으로 다양한 표현을 만들어 이 문제를 해결합니다.

> ⚠️ **자주 하는 실수**: `temperature=0`을 설정하지 않으면 LLM이 매번 다른 쿼리를 생성해 결과가 일관되지 않아요. 쿼리 생성 LLM은 반드시 temperature=0으로 설정하세요.

In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# MultiQueryRetriever를 위한 기본 Retriever와 LLM 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt 기반 FAISS Retriever와 ChatOpenAI LLM을 생성하세요
# 힌트: base_retriever는 k=3, LLM은 temperature=0 (일관된 쿼리 생성)
# 예상 결과: Retriever와 LLM 준비 완료 메시지 출력
# ============================================================

# 문서 및 retriever 준비
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# temperature=0: 쿼리 생성 결과를 일관되게 유지
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

print("✅ 기본 Retriever 및 LLM 준비 완료")

✅ 기본 Retriever 및 LLM 준비 완료


In [3]:
# ---------------------------------------------------
# MultiQueryRetriever 생성 및 LLM 생성 쿼리 확인
# ---------------------------------------------------

# ============================================================
# TODO: MultiQueryRetriever.from_llm()으로 생성하고 INFO 로그로 생성된 쿼리를 확인하세요
# 힌트: logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)
# 예상 결과: LLM이 생성한 여러 쿼리가 로그에 출력되고 문서가 검색됨
# ============================================================

from langchain.retrievers.multi_query import MultiQueryRetriever
import logging

# 로깅 설정 — INFO 레벨로 LLM이 생성한 쿼리 목록 확인 가능
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# MultiQueryRetriever: 원본 쿼리를 LLM이 3~5개로 재구성
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

# 검색
query = "NLP에서 임베딩은 어떤 역할을 하나요?"
docs = multi_query_retriever.invoke(query)

print(f"\n📝 원본 쿼리: {query}")
print(f"\n검색된 문서 수: {len(docs)}개\n")
print("="*80)
for i, doc in enumerate(docs, 1):
    print(f"[문서 {i}]")
    print(doc.page_content[:150] + "...")
    print("-"*80)

print("\n💡 MultiQueryRetriever의 장점:")
print("  - LLM이 자동으로 쿼리 다각화")
print("  - 다양한 표현으로 더 많은 관련 문서 발견")
print("  - 중복 자동 제거")

INFO:langchain.retrievers.multi_query:Generated queries: ['NLP에서 임베딩의 기능은 무엇인가요?  ', '자연어 처리에서 임베딩이 수행하는 역할은 무엇인지 알고 싶습니다.  ', 'NLP에서 임베딩이 중요한 이유는 무엇인가요?']



📝 원본 쿼리: NLP에서 임베딩은 어떤 역할을 하나요?

검색된 문서 수: 4개

[문서 1]
NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사용될 수 있습니다. 이러한 언어의 특성으로 인해, NLP 시스템은 텍스트의 문법적 구조를 ...
--------------------------------------------------------------------------------
[문서 2]
NLP의 발전은 사람과 컴퓨터 간의 소통 방식을 근본적으로 변화시켰습니다. 예를 들어, 자연어를 이해할 수 있는 인터페이스를 통해 사용자는 복잡한 쿼리나 명령어를 사용하지 않고도 정보를 검색하거나 서비스를 이용할 수 있게 되었습니다. 또한, 자동 번역 시스템의 발전으로...
--------------------------------------------------------------------------------
[문서 3]
Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될 수 있습니다. 예를 들어, 단어의 유사도 측정, 문장이나 문서의 벡터 표현 생성, 기계 번역, 감정 분석 등이 있습니다. 또한, 벡터 연산을 통해 단어 간의 의미적 관계를 추론하는 것이 가능해집니다. 예를 들어...
--------------------------------------------------------------------------------
[문서 4]
기계 학습 분야에 있어서, Scikit-learn은 특히 초보자에게 친화적인 학습 자원을 제공함으로써, 복잡한 이론과 알고리즘을 이해하는 데 중요한 역할을 합니다. 이러한 접근성과 범용성은 Scikit-learn을 기계 학습을 시작하는 이들에게 인기 있는 선택지로 만들...
-----------------------------------------------------------------

In [4]:
# ---------------------------------------------------
# 서브 쿼리별 문서 출처 추적 + 중복 제거 효과 확인
# ---------------------------------------------------

# MultiQueryRetriever가 내부에서 생성한 쿼리를 재현하여
# 각 쿼리가 어떤 문서를 가져왔는지 추적
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

# 서브 쿼리 직접 생성 (MultiQueryRetriever 내부 로직 재현)
sub_query_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate 3 different versions of the given user question to retrieve relevant documents from a vector database. By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of distance-based similarity search. Provide these alternative questions separated by newlines. Original question: {question}"""
)

chain = sub_query_prompt | llm
result = chain.invoke({"question": query})
sub_queries = [q.strip() for q in result.content.strip().split("\n") if q.strip()]

print(f"📝 원본 쿼리: {query}\n")
print(f"{'='*70}")
print("🔍 LLM이 생성한 서브 쿼리:")
print(f"{'='*70}")
for i, sq in enumerate(sub_queries, 1):
    print(f"  [{i}] {sq}")

# 각 서브 쿼리별 검색 결과 추적
print(f"\n{'='*70}")
print("📊 서브 쿼리별 검색 결과")
print(f"{'='*70}")

all_doc_contents = {}  # content → 어떤 쿼리에서 검색되었는지

# 원본 쿼리 포함
all_queries = [("원본", query)] + [(f"서브 {i+1}", sq) for i, sq in enumerate(sub_queries)]

for label, q in all_queries:
    results = base_retriever.invoke(q)
    print(f"\n[{label}] {q[:50]}...")
    for j, doc in enumerate(results, 1):
        key = doc.page_content[:60]
        if key not in all_doc_contents:
            all_doc_contents[key] = []
        all_doc_contents[key].append(label)
        print(f"  문서 {j}: {doc.page_content[:50]}...")

# 중복 분석
print(f"\n{'='*70}")
print("📊 중복 제거 효과 분석")
print(f"{'='*70}")
total_fetched = sum(len(v) for v in all_doc_contents.values())
unique_docs = len(all_doc_contents)
print(f"  총 검색 횟수: {total_fetched}회")
print(f"  고유 문서 수: {unique_docs}개")
print(f"  중복 제거율: {(1 - unique_docs / total_fetched) * 100:.0f}%")

print(f"\n  문서별 출처:")
for key, sources in sorted(all_doc_contents.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"  [{len(sources)}개 쿼리] {key}...")

print("\n💡 관찰 포인트:")
print("  - 여러 쿼리에서 공통으로 검색된 문서는 관련성이 높을 가능성 큼")
print("  - 특정 쿼리에서만 나온 문서는 해당 관점의 고유한 결과")
print("  - MultiQueryRetriever는 이 모든 결과를 자동으로 합치고 중복 제거")

📝 원본 쿼리: NLP에서 임베딩은 어떤 역할을 하나요?

🔍 LLM이 생성한 서브 쿼리:
  [1] NLP에서 임베딩의 기능은 무엇인가요?
  [2] 자연어 처리에서 임베딩이 수행하는 주요 역할은 무엇인지 알고 싶습니다.
  [3] NLP에서 단어 임베딩이 어떻게 활용되는지 설명해 주세요.

📊 서브 쿼리별 검색 결과

[원본] NLP에서 임베딩은 어떤 역할을 하나요?...
  문서 1: NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 ...
  문서 2: NLP의 발전은 사람과 컴퓨터 간의 소통 방식을 근본적으로 변화시켰습니다. 예를 들어, 자...
  문서 3: 기계 학습 분야에 있어서, Scikit-learn은 특히 초보자에게 친화적인 학습 자원을 ...

[서브 1] NLP에서 임베딩의 기능은 무엇인가요?...
  문서 1: NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 ...
  문서 2: NLP의 발전은 사람과 컴퓨터 간의 소통 방식을 근본적으로 변화시켰습니다. 예를 들어, 자...
  문서 3: Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될 수 있습니다. 예를 들어, 단어...

[서브 2] 자연어 처리에서 임베딩이 수행하는 주요 역할은 무엇인지 알고 싶습니다....
  문서 1: NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 ...
  문서 2: NLP의 발전은 사람과 컴퓨터 간의 소통 방식을 근본적으로 변화시켰습니다. 예를 들어, 자...
  문서 3: Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될 수 있습니다. 예를 들어, 단어...

[서브 3] NLP에서 단어 임베딩이 어떻게 활용되는지 설명해 주세요....
  문서 1: NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 ...
  문서 2: Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | LLM이 원본 쿼리 → 여러 관점의 쿼리를 자동 생성, 결과 합산 |
| 장점 | 단일 쿼리 한계 극복, 다양한 표현으로 recall(재현율) 향상 |
| 단점 | LLM 추가 호출 비용 발생, 응답 지연 증가 |
| 중복 제거 | 내부적으로 중복 문서를 자동으로 제거해줘요 |
| 로깅 확인 | `logging.getLogger("langchain...").setLevel(logging.INFO)` |

### 생성 쿼리 예시

원본 쿼리: "머신러닝에서 과적합을 방지하는 방법은?"

| 생성 쿼리 | 관점 |
|-----------|------|
| 과적합 문제를 해결하기 위한 정규화 기법은? | 기법 중심 |
| 딥러닝 모델의 일반화 성능을 높이는 전략은? | 목적 중심 |
| Train/Test 성능 차이를 줄이는 방법은? | 증상 중심 |

### 다음 노트북 예고

**07-MultiVectorRetriever** — 원본 문서 대신 요약본이나 가상 질문으로 임베딩하여 검색하고, 실제 반환은 원본 문서를 돌려주는 전략을 배워요.